In [17]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import copy
import time

from src.model import PINN
# Several different ways of generating collocation points are used in this notebook, to varying levels of success
from src.data import (
    generate_boundary_points,
    generate_initial_points,
    generate_collocation_points,
    generate_collocation_points_shock,
    generate_collocation_points_lhs)

# Import PINN model and training functions from source
from src.model_causal import PINN_causal
from src.train_causal import train_pinn_causal
# Import causal model and causal training functions
from src.train import train_pinn
from src.evaluate import relativeERROR

# Set default data type to float32
torch.set_default_dtype(torch.float)

# PyTorch random number generator
torch.manual_seed(1024)

# Numpy random number generator
np.random.seed(1024)

In [ ]:
# Define Burgers' Equation
def Burgers_initial_function(x):
    return -torch.sin(torch.pi*x)

# Generate boundary and initial points and collocation points
x_bc, y_bc = generate_boundary_points(100)
x_ic, y_ic = generate_initial_points(1000, Burgers_initial_function)
col_points = generate_collocation_points(10000)
mu = 0.01/torch.pi

def Burgers_equation(g, u, derivs, mu = mu):
    x, t = g[:,[0]], g[:,[1]]
    return derivs['u_t'] + u*derivs['u_x'] - mu*derivs['u_xx']

# we will test different training models averaged, trained on different collocation points and averaged
collocation_seeds = torch.randint(1,1000,(1,5)).tolist()
collocation_seeds_list = collocation_seeds[0]

In [ ]:
# Heat map comparison: PINN vs exact benchmark solution

import scipy.io
import numpy as np

# Import exact grid + solution
data = scipy.io.loadmat('data/burgers_shock.mat')
x_exact = data['x'].flatten()                         # Shape: (256,)
t_exact = data['t'].flatten()                          # Shape: (100,)
u_sol = (data['usol']).astype(np.float32)        # Shape: (256, 100)

# Build an empty grid using the exact grid's coordinates
X, T = np.meshgrid(x_exact, t_exact, indexing='ij')     # shape (256, 100)
grid = torch.tensor(np.stack([X.flatten(), T.flatten()], axis=1), dtype=torch.float32)


In [27]:
# Testing using normal model and random uniform sampled collocation points
results_summary = {}

for col_seed in collocation_seeds_list:
    
    # Set seed ONLY for generating the collocation points
    torch.manual_seed(col_seed)
    col_points = generate_collocation_points(10000)
    
    # Lock the network initialisation seed
    torch.manual_seed(42)
    pinn = PINN()
    
    print(f"Training with Collocation Seed: {col_seed}...")
    
    # Train the PINN
    history, final_loss = train_pinn(
        pinn, x_bc, y_bc, x_ic, y_ic, col_points,
        var_names=['x', 't'], PDE_residual_fn=Burgers_equation,
        epochs=10000
    )
    
    # Evaluate performance
    with torch.no_grad():
        predicted_values = pinn(grid).numpy().reshape(X.shape)
    
    relative_error = relativeERROR(torch.tensor(predicted_values), torch.tensor(u_sol)).item()
    results_summary[col_seed] = relative_error
    print(f"-> Relative Error: {relative_error:.4f}\n")
    

average_error = sum(results_summary.values())/len(results_summary)

print(f"Average Relative Error across all seeds: {average_error}")


Training with Collocation Seed: 943...
-> Relative Error: 0.4799

Training with Collocation Seed: 847...
-> Relative Error: 0.2365

Training with Collocation Seed: 907...
-> Relative Error: 0.0312

Training with Collocation Seed: 341...
-> Relative Error: 0.0147

Training with Collocation Seed: 528...
-> Relative Error: 0.6116

Average Relative Error across all seeds: 0.27475415356457233


In [30]:
# Testing using normal model and lhs sampled collocation points
results_summary = {}

for col_seed in collocation_seeds_list:
    
    # Set seed ONLY for generating the collocation points
    torch.manual_seed(col_seed)
    col_points = generate_collocation_points_lhs(10000)
    
    # Lock the network initialisation seed
    torch.manual_seed(42)
    pinn = PINN()
    
    print(f"Training with Collocation Seed: {col_seed}...")
    
    # Train the PINN
    history, final_loss = train_pinn(
        pinn, x_bc, y_bc, x_ic, y_ic, col_points,
        var_names=['x', 't'], PDE_residual_fn=Burgers_equation,
        epochs=10000
    )
    
    # Evaluate performance
    with torch.no_grad():
        predicted_values = pinn(grid).numpy().reshape(X.shape)
    
    relative_error = relativeERROR(torch.tensor(predicted_values), torch.tensor(u_sol)).item()
    results_summary[col_seed] = relative_error
    print(f"-> Relative Error: {relative_error:.4f}\n")
    

average_error = sum(results_summary.values())/len(results_summary)


print(f"Average Relative Error across all seeds: {average_error}")


Training with Collocation Seed: 943...
-> Relative Error: 0.2843

Training with Collocation Seed: 847...
-> Relative Error: 0.1497

Training with Collocation Seed: 907...
-> Relative Error: 0.3109

Training with Collocation Seed: 341...
-> Relative Error: 0.0139

Training with Collocation Seed: 528...
-> Relative Error: 0.0982

Average Relative Error across all seeds: 0.17139781825244427


In [ ]:
# Testing using causal model and random uniform sampled collocation points
results_summary = {}

for col_seed in collocation_seeds_list:
    
    # Set seed ONLY for generating the collocation points
    torch.manual_seed(col_seed)
    col_points = generate_collocation_points(10000)
    
    # Lock the network initialisation seed
    torch.manual_seed(42)
    pinn = PINN_causal()
    
    print(f"Training with Collocation Seed: {col_seed}...")
    
    # Train the PINN
    history, final_loss = train_pinn_causal(
        pinn, x_bc, y_bc, x_ic, y_ic, col_points,
        var_names=['x', 't'], PDE_residual_fn=Burgers_equation,
        epochs=10000, eps=0.2, n_buckets= 10
    )
    
    # Evaluate performance
    with torch.no_grad():
        predicted_values = pinn(grid).numpy().reshape(X.shape)
    
    relative_error = relativeERROR(torch.tensor(predicted_values), torch.tensor(u_sol)).item()
    results_summary[col_seed] = relative_error
    print(f"-> Relative Error: {relative_error:.4f}\n")
    

average_error = sum(results_summary.values())/len(results_summary)


print(f"Average Relative Error across all seeds: {average_error}")

Training with Collocation Seed: 943...
-> Relative Error: 0.1106

Training with Collocation Seed: 847...
-> Relative Error: 0.0277

Training with Collocation Seed: 907...
-> Relative Error: 0.0226

Training with Collocation Seed: 341...
-> Relative Error: 0.0629

Training with Collocation Seed: 528...
-> Relative Error: 0.0587

Average Relative Error across all seeds: 0.05651624947786331


In [ ]:
# Testing using causal model and lhs sampled collocation points
results_summary = {}

for col_seed in collocation_seeds_list:
    
    # Set seed ONLY for generating the collocation points
    torch.manual_seed(col_seed)
    col_points = generate_collocation_points_lhs(10000)
    
    # Lock the network initialisation seed
    torch.manual_seed(42)
    pinn = PINN_causal()
    
    print(f"Training with Collocation Seed: {col_seed}...")
    
    # Train the PINN
    history, final_loss = train_pinn_causal(
        pinn, x_bc, y_bc, x_ic, y_ic, col_points,
        var_names=['x', 't'], PDE_residual_fn=Burgers_equation,
        epochs=10000, eps=0.2, n_buckets= 10
    )
    
    # Evaluate performance
    with torch.no_grad():
        predicted_values = pinn(grid).numpy().reshape(X.shape)
    
    relative_error = relativeERROR(torch.tensor(predicted_values), torch.tensor(u_sol)).item()
    results_summary[col_seed] = relative_error
    print(f"-> Relative Error: {relative_error:.4f}\n")
    

average_error = sum(results_summary.values())/len(results_summary)


print(f"Average Relative Error across all seeds: {average_error}")

Training with Collocation Seed: 943...
-> Relative Error: 0.1230

Training with Collocation Seed: 847...
-> Relative Error: 0.1136

Training with Collocation Seed: 907...
-> Relative Error: 0.1133

Training with Collocation Seed: 341...
-> Relative Error: 0.0329

Training with Collocation Seed: 528...
-> Relative Error: 0.0506

Average Relative Error across all seeds: 0.08666796237230301


From this testing we can see that on their own lhs sampling (reduced to 17% from 27.5%) and causal training (reduced to 5.7%) both improve the relative L2 error for the Burgers equation. However, when used in combination (error reduced to 8.7%) the error was actually worse than just causal training alone. Further testing is required to understand the true meaning behind this. Note - there was a large variation accross different collocation seeds, showing that creating useful collocation points is incredibly important for PINN training especially for this example. 